# DataLoader

---
## 先认识术语: 
- 样本(sample): 数据里的一条具体记录. Fashion-MNIST里, 一张衣服图片就是一个样本. 
- 特征(feature): 模型用来做判断的输入. 对这张图片来说, 特征就是图片的像素值. 
- 数据集(Dataset): 提供样本和标签的数据对象. 可以按编号取一条数据, 例如取第0条. 

In [ ]:
image, label = training_data[0]

这行代码取出一条样本: image是图片特征, label是它的类别编号. \
\
要点是: Dataset[i]通常给你一个样本及其标签. 它提供按编号读取数据的方式; 数据具体是在内存里还是读取时从文件加载, 取决于Dataset的实现. 

---
## 自己写一个Dataset
先把代码里的几个词说清楚: 
- 类(class): 创建某种对象的"蓝图"
- 方法(method): 写在类里面的函数. 
- 继承(inheritance): class MyDataset(Dataset)表示自定义类沿用Pytorch的Dataset约定. 
- serf: 指当前这个Dataset对象本身. 
- 索引(index): 用来指出第几条样本编号; 这里的idx就是这个编号. 

自定义Dataset最重要的是三个方法. 可以先把它们理解成: 初始化清单, 报告数量, 按编号取一条样本. 

In [1]:
import torch
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self):
        self.samples = [
            (torch.tensor([0.2, 0.8]), 1), 
            (torch.tensor([0.5, 0.5]), 0), 
            (torch.tensor([0.9, 0.1]), 1), 
        ]
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

逐个看它们做什么: 
1. \__init\__: 准备Dataset\
创建对象时运行一次. 这里把三条(特征, 标签)存到self.samples. 真实项目中, 通常会在这里记下图片目录, 标签文件等信息. 
2. \__len\__: 告诉PyTorch有多少条\
len(dataset)会调用它. 上面的例子会得到3, 表示有3条样本. 
3. \__getitem\__: 按编号取一条\
dataset[1]会调用\__getitem\__(1), 返回标号为1的那一条. Python从0开始编号, 所以它会取列表里的第二条. 

在真实图片Dataset中, \__getitem\__通常还要根据编号找到图片文件, 读取图片, 再取出对应标签, 最后返回(image, label).

---
## 批次, 形状, 最后一批和打乱顺序
### 1. 什么是batch
批次(batch, 也叫mini-batch): 一次交给模型处理的以小组样本. Dataset每次取一条, DataLoader则把多条样本整理成一批. 

In [ ]:
train_dataloader = Dataset(
    training_data, 
    batch_size=64, 
    shuffle=True
)

这里batch_size=64表示每批最多放64条样本. 训练代码通常会逐批读取:

In [ ]:
for feature, labels in train_dataloader:
    # 这里拿到一批图片和对应标签
    ...

### 2. batch会怎样改变形状?
单张FashionMNIST图片的形状是[1, 28, 28]: 1个灰度通道, 高和宽都是28. \
\
64张图片合在一批后, DataLoader会把"样本数量"放在最前面: 

In [1]:
features: [64, 1, 28, 28]
labels: [64]

### 3. 最后一批为什么可能不满?
轮次(epoch): 模型把整个训练集完整过一遍, 成为一个epoch. \
\
FashionMINIST训练集有60,000条. 若batch_size=64: 
- 937批$\times$64条 = 59,968条
- 还剩32条, 所以最后一批有32条
- 默认drop_last=False, 这32条也会保留, 因此总共938批
- 若默认drop_last=True, 最后不足64条的批次会被丢掉

### 4. shuffle=True是什么？
**打乱(shuffle)**: 每轮训练开始时, 改变样本进入批次的顺序. 图片与标签仍然配对, 不会把图片和别人的标签错配. 训练时通常设为True, 让模型不总是按固定顺序看到样本; 验证或测试时通常设为False, 便于保持顺序和复现结果. 官方基础教程的测试示例也写了shuffle=True, 但这不是错误, 只是测试时一般没必要打乱. \


---
## 数据集划分
- **训练集 (training set)**: 模型用来学习的数据. 
- **验证集 (validation set)**: 开发过程中用来检查模型, 比较方案的数据. 
- **测试集 (test set)**: 留到最后评估模型表现的数据. 

FashionMNIST的train=True表示使用它提供的训练集, train=False表示使用它提供的测试集. 注意: 教程视频把train=False的数据变量命名为validation_set; 变量名只是名字, 实际加载的仍是FashionMNIST的测试划分. 更严谨的项目通常会从训练数据中单独留出验证集, 测试集则留到最后. 

### Transforms: 让数据变成模型需要的样子
**数据变换 (transform)**: 读取样本时, 对图片或标签做预处理. 官方教程里用Compose把多个变换按顺序串起来: 

In [ ]:
transform = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
])

这里ToImage()把图片转成PyTorch可处理的图像张量; ToDtype(..., scale=True)把像素转成float32, 并将像素值缩放到[0, 1\]. 例如原先范围为0-255的像素值会缩放到0-1.\
\
transform用于图片等特征; target_transform用于标签. 当前我们只给图片加transform, 标签仍保持整数类别编号, 比如9.

### 把它们连起来

In [ ]:
training_data = datasets.FashionMNIST(
    root="data", 
    train=True, 
    download=True, 
    transform=transform, 
)

test_data = datasets.FashionMNIST(
    root="data", 
    train=False, 
    download=True, 
    transform=transform, 
)

train_loader = DataLoader(training_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

整个过程可以这样记:\
\
Dataset: 定义一条样本是什么, 以及如何取出它\
\
DataLoader: 从Dataset取样本, 组成batch\
\
transform: 每次取图片时, 把图片预处理成需要的格式

如果单张图片形状是[1, 28, 28\], 训练批次大小是64, 那么图片批次形状就是[64, 1, 28, 28\]. transform改变的是图片内容的表示方式, 不会把一张图片变成64张.